In [1]:
from pathlib import Path
import torch
import requests
import time

In [2]:
# --- Benchmark PDB list (all splits) ---
repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve()
data_dir = repo_root / "voxbind" / "dataset" / "data"

benchmark_pdbs = set()
for pt_path in sorted(data_dir.glob("data_*.pt")):
    samples = torch.load(pt_path, weights_only=False)
    for pocket_dict, _ in samples:
        pdb_id = pocket_dict["id"].split("/")[1].split("_")[0].lower()
        benchmark_pdbs.add(pdb_id)

pdb_list = sorted(benchmark_pdbs)
print(f"Total benchmark PDB IDs (all splits): {len(pdb_list):,}")
print("Example:", pdb_list[:10])

Total benchmark PDB IDs (all splits): 14,735
Example: ['132l', '135l', '14gs', '1a0g', '1a18', '1a26', '1a27', '1a28', '1a2d', '1a2g']


In [3]:
def check_density_availability(pdb_id):
    """Check whether experimental density data is available for a PDB entry.

    Strategy
    --------
    Summary endpoint → experimental method:
      - Electron Microscopy : look for EMDB accession in related_structures
                              (field "resource" == "EMDB")
      - X-ray diffraction   : query electron_density_statistics endpoint;
                              HTTP 200 means EDS 2mFo-DFc/mFo-DFc maps exist
                              (structure factors were deposited).
                              Many older structures (pre-~2000) never deposited
                              SFs, so they correctly return 404 here.

    Endpoints used (EBI only, RCSB not reachable from this network)
    ---------------------------------------------------------------
    Summary : https://www.ebi.ac.uk/pdbe/api/pdb/entry/summary/{pdb_id}
    EDS     : https://www.ebi.ac.uk/pdbe/api/pdb/entry/electron_density_statistics/{pdb_id}
    """
    pdb_id = pdb_id.lower()

    try:
        # -- Step 1: summary ------------------------------------------------
        r = requests.get(
            f"https://www.ebi.ac.uk/pdbe/api/pdb/entry/summary/{pdb_id}",
            timeout=15,
        )
        if r.status_code != 200:
            return {"pdb_id": pdb_id, "has_real_density": False,
                    "source": None, "source_id": None, "method": "not_found"}

        data    = r.json().get(pdb_id, [])[0]
        methods = data.get("experimental_method", [])
        methods_upper = [m.upper() for m in methods]

        result = {
            "pdb_id":          pdb_id,
            "method":          ", ".join(methods),
            "has_real_density": False,
            "source":          None,
            "source_id":       None,
        }

        # -- Step 2a: cryo-EM -----------------------------------------------
        if "ELECTRON MICROSCOPY" in methods_upper:
            related      = data.get("related_structures", [])
            emdb_entries = [s for s in related if s.get("resource") == "EMDB"]
            if emdb_entries:
                result["has_real_density"] = True
                result["source"]           = "EMDB"
                result["source_id"]        = emdb_entries[0].get("accession")

        # -- Step 2b: X-ray -------------------------------------------------
        elif "X-RAY DIFFRACTION" in methods_upper:
            # electron_density_statistics → 200 iff structure factors deposited
            eds = requests.get(
                f"https://www.ebi.ac.uk/pdbe/api/pdb/entry/electron_density_statistics/{pdb_id}",
                timeout=15,
            )
            if eds.status_code == 200:
                result["has_real_density"] = True
                result["source"]           = "PDBe EDS"
                result["source_id"]        = pdb_id

        return result

    except Exception as e:
        return {"pdb_id": pdb_id, "has_real_density": False,
                "source": None, "source_id": None, "method": f"error: {e}"}

In [4]:
# --- Density check run ---
# Set MAX_TO_QUERY=None to run all (can take a long time).
MAX_TO_QUERY = 1
# query_list = pdb_list if MAX_TO_QUERY is None else pdb_list[:MAX_TO_QUERY]
query_list = ["14gs"]

for pdb in query_list:
    info = check_density_availability(pdb)
    print(f"[{info['pdb_id'].upper()}] Method: {info.get('method', 'N/A')}")
    if info.get('has_real_density'):
        print(f"  -> Density Found! Source: {info['source']} ({info['source_id']})")
    else:
        print("  -> No real density found. Use pdb2vol proxy.")
    time.sleep(0.5)  # API etiquette

[14GS] Method: X-ray diffraction
  -> Density Found! Source: PDBe EDS (14gs)


In [5]:
# --- structure_factors coverage over all benchmark PDB IDs ---
from tqdm.auto import tqdm

session = requests.Session()

has_structure_factors = []
no_structure_factors = []
errors = []

for pdb_id in tqdm(pdb_list, desc="Checking PDBe structure_factors", unit="pdb"):
    files_url = f"https://www.ebi.ac.uk/pdbe/api/pdb/entry/files/{pdb_id}"
    try:
        resp = session.get(files_url, timeout=20)
        if resp.status_code != 200:
            errors.append((pdb_id, f"HTTP {resp.status_code}"))
            continue

        files_data = resp.json().get(pdb_id, {})
        sf = files_data.get("structure_factors")

        if sf:
            has_structure_factors.append(pdb_id)
        else:
            no_structure_factors.append(pdb_id)

    except Exception as exc:
        errors.append((pdb_id, str(exc)))

    time.sleep(0.05)  # gentle API pacing

n_total = len(pdb_list)
n_has_sf = len(has_structure_factors)
n_no_sf = len(no_structure_factors)
n_err = len(errors)

pct_has_sf = (100.0 * n_has_sf / n_total) if n_total else 0.0

print("\n=== PDBe structure_factors Coverage (Benchmark, all splits) ===")
print(f"Total benchmark PDB IDs: {n_total:,}")
print(f"Has structure_factors: {n_has_sf:,} ({pct_has_sf:.2f}%)")
print(f"No structure_factors: {n_no_sf:,}")
print(f"Request errors: {n_err:,}")
print("Example PDBs with structure_factors:", has_structure_factors[:20])

Checking PDBe structure_factors:   0%|          | 0/14735 [00:00<?, ?pdb/s]


=== PDBe structure_factors Coverage (Benchmark, all splits) ===
Total benchmark PDB IDs: 14,735
Has structure_factors: 0 (0.00%)
No structure_factors: 14,702
Request errors: 33
Example PDBs with structure_factors: []
